
# Late Delivery Prediction — Machine Learning

## Project Overview

This project develops a **binary classification model** to predict whether an e-commerce order will arrive late using the Brazilian Olist e-commerce dataset.

The goal is to identify orders at risk of late delivery using information available **at the time of purchase**, helping businesses improve delivery reliability and customer satisfaction.

### Objectives
- Prepare and explore historical order data from PostgreSQL.
- Engineer predictive features while preventing data leakage.
- Train and compare Logistic Regression and Random Forest classifiers.
- Evaluate model performance using precision, recall, F1-score, and confusion matrices.

### Prediction Target
- **0 — Not Late:** Delivered on or before the estimated delivery date.
- **1 — Late:** Delivered after the estimated delivery date.

Orders with missing delivery outcomes will be excluded from training.


## 1. Data Extraction

Load historical order data from the PostgreSQL `retail_bi` database using Pandas and SQLAlchemy.

We'll begin with order-level information and our binary prediction target (`is_late`). Additional features will be introduced during feature engineering.

In [39]:
# Install the PostgreSQL driver into the notebook's Python environment, psycopg2 handles PostgreSQL communication
# %pip install psycopg2-binary

import os
import pandas as pd
# SQLAlchemy manages database connections
from sqlalchemy import create_engine, text
from getpass import getpass

# Prompt for the database password without displaying it in the notebook
db_password = getpass("PostgreSQL password: ")

# Create a connection to our local PostgreSQL database
engine = create_engine(
    "postgresql+psycopg2://",
    connect_args={
        "host": "localhost",
        "port": 5432,
        "dbname": "retail_bi",
        "user": "postgres",
        "password": db_password
    }
)

# Execute a simple SQL query to verify the database connection
with engine.connect() as connection:
    result = connection.execute(text("SELECT 1"))
    print("Connection successful:", result.scalar())

Connection successful: 1


In [40]:
# SQL query to retrieve historical orders and create our prediction target
query = """
SELECT
    order_id,
    customer_id,
    order_purchase_timestamp,
    order_estimated_delivery_date,
    delivery_status,

    -- Binary classification target: 1 = Late, 0 = Not Late
    CASE
        WHEN delivery_status = 'Late' THEN 1
        ELSE 0
    END AS is_late

FROM orders

-- Exclude orders with missing delivery outcomes
WHERE delivery_status IN ('Early', 'On Time', 'Late')
"""

# Execute the SQL query and load the results into a Pandas DataFrame
df = pd.read_sql(query, engine)

# Preview the first five rows
print(df.head())

# Display the number of rows and columns in our dataset
print("Dataset shape:", df.shape)

# Count how many orders belong to each target class
print("\nTarget distribution:")
print(df["is_late"].value_counts())

                           order_id                       customer_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   
2  47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089   
3  949d5b44dbf5de918fe9c16f97b45f8a  f88197465ea7920adcdbec7375364d82   
4  ad21c59c0840e6cb83a9ceb5573f8159  8ab97904e6daea8866dbdbc4fb7aad2c   

  order_purchase_timestamp order_estimated_delivery_date delivery_status  \
0      2017-10-02 10:56:33                    2017-10-18           Early   
1      2018-07-24 20:41:37                    2018-08-13           Early   
2      2018-08-08 08:38:49                    2018-09-04           Early   
3      2017-11-18 19:28:06                    2017-12-15           Early   
4      2018-02-13 21:18:39                    2018-02-26           Early   

   is_late  
0        0  
1        0  
2        0  
3        0  
4        0  
Dataset shape: (96476, 6)


In [41]:
# Count missing values in each column
print("Missing values:")
print(df.isna().sum())

# Check whether any order_id appears more than once
print("\nDuplicate orders:", df["order_id"].duplicated().sum())

# Inspect the data types of our columns
print("\nColumn data types:")
print(df.dtypes)


Missing values:
order_id                         0
customer_id                      0
order_purchase_timestamp         0
order_estimated_delivery_date    0
delivery_status                  0
is_late                          0
dtype: int64

Duplicate orders: 0

Column data types:
order_id                                    str
customer_id                                 str
order_purchase_timestamp         datetime64[us]
order_estimated_delivery_date    datetime64[us]
delivery_status                             str
is_late                                   int64
dtype: object


## 2. Feature Engineering

Create predictive features using information available at the time of purchase.

### Purchase Timing & Delivery Estimates
- Extract purchase month, day of week, and hour.
- Calculate the estimated delivery window in days.

### Customer Location
- Retrieve customer state information from PostgreSQL.
- Merge customer data with our order-level dataset using `customer_id`.
- Check for missing values after the merge.

Customer location may help identify regional differences in delivery reliability.

In [42]:
# Extract the month when the customer placed the order (1–12)
df["purchase_month"] = df["order_purchase_timestamp"].dt.month

# Extract the day of the week (Monday = 0, Sunday = 6)
df["purchase_day_of_week"] = df["order_purchase_timestamp"].dt.dayofweek

# Extract the hour when the order was placed (0–23)
df["purchase_hour"] = df["order_purchase_timestamp"].dt.hour

# Calculate the number of days between purchase and estimated delivery
df["estimated_delivery_days"] = (
    df["order_estimated_delivery_date"] - df["order_purchase_timestamp"]
).dt.total_seconds() / 86400 # converts seconds into days.

# Preview our new features alongside the prediction target
df[
    [
        "purchase_month",
        "purchase_day_of_week",
        "purchase_hour",
        "estimated_delivery_days",
        "is_late"
    ]
].head()


,purchase_month,purchase_day_of_week,purchase_hour,estimated_delivery_days,is_late
0,10,0,10,15.544063,0
1,7,1,20,19.137766,0
2,8,2,8,26.639711,0
3,11,5,19,26.188819,0
4,2,1,21,12.112049,0


In [43]:
# Retrieve customer IDs and their corresponding states from PostgreSQL
customers_query = """
SELECT
    customer_id,
    customer_state
FROM customers
"""

# Load the customer data into a separate DataFrame
customers_df = pd.read_sql(customers_query, engine)

# Merge customer states into our ML dataset using customer_id
# how="left" keeps every order from our original dataset
# validate="many_to_one" checks that each customer_id matches at most one customer record
df = df.merge(
    customers_df,
    on="customer_id",
    how="left",
    validate="many_to_one"
)

# Check whether any orders are missing customer state information
print("Missing customer states:", df["customer_state"].isna().sum())

# Preview our new feature
df[["customer_id", "customer_state", "is_late"]].head()


Missing customer states: 0


,customer_id,customer_state,is_late
0,9ef432eb6251297304e76186b10a928d,SP,0
1,b0830fb4747a6c6d20dea0b8c802d7ef,BA,0
2,41ce2a54c0b03bf3443c3d931a367089,GO,0
3,f88197465ea7920adcdbec7375364d82,RN,0
4,8ab97904e6daea8866dbdbc4fb7aad2c,SP,0


### Order-Level Features

Aggregate data from `fact_order_items` to create two additional predictive features:

- **`total_order_value`** — Total merchandise value of each order.
- **`total_items`** — Number of items purchased per order.

Since `fact_order_items` contains multiple rows per order, we use SQL aggregation (`GROUP BY order_id`) before merging to preserve **one row per order** in our ML dataset.

These features may help identify relationships between order size, purchase value, and delivery delays.

In [44]:
# Aggregate order items into one row per order
order_items_query = """
SELECT
    order_id,
    SUM(price) AS total_order_value,
    COUNT(*) AS total_items
FROM fact_order_items
GROUP BY order_id
"""

# Load aggregated order information from PostgreSQL
order_items_df = pd.read_sql(order_items_query, engine)

# Merge order-level features into our ML dataset
# validate="one_to_one" ensures neither DataFrame has duplicate order IDs
df = df.merge(
    order_items_df,
    on="order_id",
    how="left",
    validate="one_to_one"
)

# Check whether any orders are missing item information
print("Missing order values:", df["total_order_value"].isna().sum())
print("Missing item counts:", df["total_items"].isna().sum())

# Confirm the number of rows hasn't changed
print("Dataset shape:", df.shape)

# Preview the new features
df[["order_id", "total_order_value", "total_items", "is_late"]].head()

# Verify that every row represents a unique order
print("Duplicate orders:", df["order_id"].duplicated().sum())



Missing order values: 0
Missing item counts: 0
Dataset shape: (96476, 13)
Duplicate orders: 0


### Seller-Level Features

Investigate seller information to identify potential features related to shipping origin and delivery performance.

- Count the number of unique sellers per order using `COUNT(DISTINCT seller_id)`.
- Determine how frequently orders contain products from multiple sellers.
- Evaluate how seller location can be incorporated while preserving **one row per order**.

Seller location may help capture geographic factors affecting delivery times and delays.

In [45]:
# Count the number of unique sellers associated with each order
seller_query = """
SELECT
    order_id,
    COUNT(DISTINCT seller_id) AS total_sellers
FROM fact_order_items
GROUP BY order_id
"""

# Load seller counts from PostgreSQL
seller_counts_df = pd.read_sql(seller_query, engine)

# Count how many orders have 1, 2, 3, etc. sellers
print("Number of sellers per order:")
print(seller_counts_df["total_sellers"].value_counts().sort_index())

Number of sellers per order:
total_sellers
1    97388
2     1219
3       54
4        3
5        2
Name: count, dtype: int64


#### Number of Sellers per Order

Approximately **98.7% of orders contain products from a single seller**, while 1.3% involve multiple sellers.

- Merge `total_sellers` into the ML dataset using `order_id`.
- Preserve **one row per order** to prevent duplicate training records.
- Validate the merge by checking for missing values and duplicate orders.

The number of sellers may help capture differences in order fulfillment complexity and delivery risk.

In [46]:
# Merge the number of unique sellers into our ML dataset
df = df.merge(
    seller_counts_df,
    on="order_id",
    how="left",
    validate="one_to_one"
)

# Verify the merge didn't introduce missing values or duplicate orders
print("Missing seller counts:", df["total_sellers"].isna().sum())
print("Duplicate orders:", df["order_id"].duplicated().sum())

# Preview our new feature
df[["order_id", "total_sellers", "is_late"]].head()

Missing seller counts: 0
Duplicate orders: 0


,order_id,total_sellers,is_late
0,e481f51cbdc54678b7cc49136f2d6af7,1,0
1,53cdb2fc8bc7dce0b6741e2150273451,1,0
2,47770eb9100c2d0c44946d9cf07ec65d,1,0
3,949d5b44dbf5de918fe9c16f97b45f8a,1,0
4,ad21c59c0840e6cb83a9ceb5573f8159,1,0


In [47]:
# Retrieve seller location information aggregated by order
# ARRAY_AGG(DISTINCT s.seller_state) collects unique seller states into an array for each order.
seller_location_query = """
SELECT
    f.order_id,
    COUNT(DISTINCT s.seller_state) AS total_seller_states,
    ARRAY_AGG(DISTINCT s.seller_state) AS seller_states
FROM fact_order_items f
JOIN sellers s
    ON f.seller_id = s.seller_id
GROUP BY f.order_id
"""

# Execute the query and load the results into a DataFrame
seller_location_df = pd.read_sql(seller_location_query, engine)

# Merge seller locations into our order-level ML dataset
df = df.merge(
    seller_location_df,
    on="order_id",
    how="left",
    validate="one_to_one"
)

# Check for missing values and duplicate orders
print("Missing seller locations:", df["seller_states"].isna().sum())
print("Duplicate orders:", df["order_id"].duplicated().sum())

# Preview seller locations alongside customer states
print(df[["customer_state", "seller_states", "total_seller_states"]].head())

Missing seller locations: 0
Duplicate orders: 0
  customer_state seller_states  total_seller_states
0             SP          [SP]                    1
1             BA          [SP]                    1
2             GO          [SP]                    1
3             RN          [MG]                    1
4             SP          [SP]                    1


In [49]:
# Check whether every seller state matches the customer's state
# all() returns True only if every comparison is True
df["same_state_shipping"] = df.apply(
    # function lambda takes a row, returns int value of true(1) or false(0)
    lambda row: int(
        # Returns true if all the seller states matches customer state (obvs won't if theres more than 1 seller state)
        all(
            
            state == row["customer_state"] 
            # for loop to check every state in seller_state since that is an array
            for state in row["seller_states"]
        )
    ),
    axis=1 # axis=1 - apply the function to each row, which is why we can use row as a varible in our lambda function
) 

# Count orders in each shipping category
print(df["same_state_shipping"].value_counts())

# Preview our new feature
df[["customer_state", "seller_states", "same_state_shipping", "is_late"]].head()


same_state_shipping
0    61899
1    34577
Name: count, dtype: int64


,customer_state,seller_states,same_state_shipping,is_late
0,SP,[SP],1,0
1,BA,[SP],0,0
2,GO,[SP],0,0
3,RN,[MG],0,0
4,SP,[SP],1,0


In [50]:
# Compare late delivery rates for same-state vs different-state shipping
df.groupby("same_state_shipping")["is_late"].agg(["count", "sum", "mean"])

,count,sum,mean
same_state_shipping,,,
0,61899,4972,0.080324
1,34577,1563,0.045203


#### Same-State Shipping

Created `same_state_shipping` to identify whether all sellers are located in the same state as the customer.

- `1`: All sellers are in the customer's state.
- `0`: At least one seller is in a different state.

**Findings:**
- Different-state orders: 61,899 orders, with an **8.03%** late delivery rate.
- Same-state orders: 34,577 orders, with a **4.52%** late delivery rate.
- Different-state orders were approximately **1.78 times more likely** to arrive late.

This suggests that seller-customer location differences may be a useful predictor of late deliveries.

In [48]:
# Close all idle database connections managed by the engine
engine.dispose()